# TReconLM Cross-Dataset Experiments

Evaluate how models fine-tuned on one dataset perform on another.

**Experiments:**
1. **In-domain baselines**, each fine-tuned model on its own test set
2. **Cross-dataset**, each fine-tuned model on every other test set
3. **Pretrained models**, pretrained (no fine-tuning) on all test sets
4. **Length-controlled**, Chandak cropped to 110nt and 60nt, Microsoft cropped to 60nt, DNAformer cropped to 110nt, to isolate length vs error distribution effects
5. **DNAformer datasets** (FC1, FC2, Pilot), each treated as separate dataset (128nt, Nanopore)

**Datasets:** Chandak (117nt), Microsoft (110nt), Noisy DNA (60nt), DNAformer FC1/FC2/Pilot (128nt)

Run the data preparation notebooks first:
- `../data/microsoft_data/microsoft_data.ipynb`
- `../data/noisy_dna/noisyDNA_data.ipynb`
- `../data/chandak/README.md`
- `../data/dnaformer/remove_index.py --all` (convert binned format to standard)

## 1. Setup

In [1]:
%matplotlib inline
import os
import sys
import gc
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from huggingface_hub import hf_hub_download

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# Import project modules
from src.gpt_pkg.model import GPT, GPTConfig
from src.eval_pkg.GPT_Inference import GPT_Inference
from src.utils.helper_functions import filter_string
from src.utils.hamming_distance import hamming_distance_postprocessed
from Levenshtein import distance as levenshtein_distance

# Import tutorial utilities
from tutorial.utils import get_model_info, load_vocabulary

# Import alignment helper from error model estimation
sys.path.insert(0, str(project_root / 'data' / 'error_model'))
from estimate_error_model import align_read_to_gt

print("Setup complete!")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def crop_read_to_length(read, gt, target_length):
    """Crop a read to match the first target_length positions of the ground truth, using alignment."""
    ops = align_read_to_gt(read, gt)
    gt_pos = 0
    read_pos = 0
    for op, gt_base, read_base in ops:
        if gt_pos >= target_length:
            break
        if op in ('=', 'X'):
            gt_pos += 1
            read_pos += 1
        elif op == 'D':
            gt_pos += 1
        elif op == 'I':
            read_pos += 1
    return read[:read_pos]

def crop_dataset(reads_path, gt_path, crop_length, output_dir, min_cs=2, max_cs=10):
    """Crop a dataset to a target GT length using alignment-based read trimming."""
    with open(reads_path, 'r') as f:
        content = f.read()
    clusters_raw = [c.strip().split('\n') for c in content.split('===============================') if c.strip()]

    with open(gt_path, 'r') as f:
        gts = [line.strip() for line in f if line.strip()]

    cropped_clusters = []
    cropped_gts = []

    for cluster, gt in tqdm(zip(clusters_raw, gts), total=len(gts), desc=f'Cropping to {crop_length}nt'):
        if not (min_cs <= len(cluster) <= max_cs):
            continue
        cropped_reads = [crop_read_to_length(read, gt, crop_length) for read in cluster]
        cropped_clusters.append(cropped_reads)
        cropped_gts.append(gt[:crop_length])

    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    with open(output_dir / 'reads.txt', 'w') as f:
        for i, cluster in enumerate(cropped_clusters):
            for read in cluster:
                f.write(read + '\n')
            if i < len(cropped_clusters) - 1:
                f.write('===============================\n')

    with open(output_dir / 'ground_truth.txt', 'w') as f:
        for gt in cropped_gts:
            f.write(gt + '\n')

    avg_read_len = np.mean([len(r) for c in cropped_clusters for r in c])
    print(f'  Clusters: {len(cropped_clusters)}, GT length: {crop_length}nt, Avg read length: {avg_read_len:.1f}nt')



Setup complete!
CUDA available: True
GPU: NVIDIA L40


## 2. Create Length-Controlled Data

Crop datasets using alignment-based trimming to isolate length vs error distribution effects.
For each read, we align it to the ground truth, then crop at the target position.

- **Chandak (117nt)** cropped to **110nt** and **60nt**
- **Microsoft (110nt)** cropped to **60nt**
- **DNAformer FC1/FC2/Pilot (128nt)** cropped to **110nt** and **60nt**

This allows fair comparison with models trained at different sequence lengths.

In [2]:
# Crop Chandak data to 110nt
print("Chandak (117nt) -> 110nt:")
crop_dataset(
    '../data/chandak/data_chandak/reads.txt',
    '../data/chandak/data_chandak/ground_truth.txt',
    crop_length=110,
    output_dir='cropped_chandak_110',
)

Chandak (117nt) -> 110nt:


Cropping to 110nt:   0%|          | 0/8481 [00:00<?, ?it/s]

Cropping to 110nt: 100%|██████████| 8481/8481 [00:03<00:00, 2525.41it/s]


  Clusters: 8481, GT length: 110nt, Avg read length: 112.1nt


In [3]:
# Crop Chandak data to 60nt
print("Chandak (117nt) -> 60nt:")
crop_dataset(
    '../data/chandak/data_chandak/reads.txt',
    '../data/chandak/data_chandak/ground_truth.txt',
    crop_length=60,
    output_dir='cropped_chandak_60',
)

Chandak (117nt) -> 60nt:


Cropping to 60nt:   0%|          | 0/8481 [00:00<?, ?it/s]

Cropping to 60nt: 100%|██████████| 8481/8481 [00:03<00:00, 2775.72it/s]

  Clusters: 8481, GT length: 60nt, Avg read length: 61.5nt


In [4]:
# Crop Microsoft data to 60nt
print("Microsoft (110nt) -> 60nt:")
crop_dataset(
    '../data/microsoft_data/data_microsoft/reads.txt',
    '../data/microsoft_data/data_microsoft/ground_truth.txt',
    crop_length=60,
    output_dir='cropped_microsoft_60',
)

Microsoft (110nt) -> 60nt:


Cropping to 60nt: 100%|██████████| 5109/5109 [00:01<00:00, 3735.96it/s]

  Clusters: 5109, GT length: 60nt, Avg read length: 59.8nt


In [5]:
# Crop DNAformer data (128nt) to 110nt and 60nt for length-controlled experiments
dnaformer_datasets = {
    'BinnedNanoporeFirstFlowcell': 'DNAformer FC1',
    'BinnedNanoporeSecondFlowcell': 'DNAformer FC2',
    'BinnedPilotNanopore': 'DNAformer Pilot',
}

for folder_name, label in dnaformer_datasets.items():
    reads_path = f'../data/dnaformer/without_index_data/{folder_name}/reads.txt'
    gt_path = f'../data/dnaformer/without_index_data/{folder_name}/ground_truth.txt'
    for crop_len in [110, 60]:
        output_dir = f'cropped_dnaformer_{folder_name}_{crop_len}'
        print(f"{label} (128nt) -> {crop_len}nt:")
        crop_dataset(reads_path, gt_path, crop_length=crop_len, output_dir=output_dir)


DNAformer FC1 (128nt) -> 110nt:


Cropping to 110nt: 100%|██████████| 232816/232816 [01:14<00:00, 3120.47it/s]


  Clusters: 232816, GT length: 110nt, Avg read length: 109.5nt
DNAformer FC1 (128nt) -> 60nt:


Cropping to 60nt: 100%|██████████| 232816/232816 [01:07<00:00, 3465.61it/s]


  Clusters: 232816, GT length: 60nt, Avg read length: 59.7nt
DNAformer FC2 (128nt) -> 110nt:


...

  Clusters: 157461, GT length: 60nt, Avg read length: 59.7nt
DNAformer Pilot (128nt) -> 110nt:


Cropping to 110nt: 100%|██████████| 126377/126377 [00:45<00:00, 2803.00it/s]


  Clusters: 126377, GT length: 110nt, Avg read length: 109.2nt
DNAformer Pilot (128nt) -> 60nt:


Cropping to 60nt: 100%|██████████| 126377/126377 [00:40<00:00, 3123.91it/s]


  Clusters: 126377, GT length: 60nt, Avg read length: 59.6nt


## 3. Define Experiments

Note: `reads.txt` and `ground_truth.txt` contain the test split only.

DNAformer data was converted from binned format using `data/dnaformer/remove_index.py`.

In [6]:
cross_experiments = [
    # =============================================================
    # Chandak data (original 117nt)
    # =============================================================
    {
        'name': 'Chandak model on Chandak data',
        'reads_path': '../data/chandak/data_chandak/reads.txt',
        'gt_path': '../data/chandak/data_chandak/ground_truth.txt',
        'seq_length': 117,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Microsoft model on Chandak data',
        'reads_path': '../data/chandak/data_chandak/reads.txt',
        'gt_path': '../data/chandak/data_chandak/ground_truth.txt',
        'seq_length': 117,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Noisy DNA model on Chandak data',
        'reads_path': '../data/chandak/data_chandak/reads.txt',
        'gt_path': '../data/chandak/data_chandak/ground_truth.txt',
        'seq_length': 117,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 110nt on Chandak data',
        'reads_path': '../data/chandak/data_chandak/reads.txt',
        'gt_path': '../data/chandak/data_chandak/ground_truth.txt',
        'seq_length': 117,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Pretrained 60nt on Chandak data',
        'reads_path': '../data/chandak/data_chandak/reads.txt',
        'gt_path': '../data/chandak/data_chandak/ground_truth.txt',
        'seq_length': 117,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on Chandak data',
        'reads_path': '../data/chandak/data_chandak/reads.txt',
        'gt_path': '../data/chandak/data_chandak/ground_truth.txt',
        'seq_length': 117,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # Chandak-110 data (cropped to 110nt)
    # =============================================================
    {
        'name': 'Chandak model on Chandak-110 data',
        'reads_path': 'cropped_chandak_110/reads.txt',
        'gt_path': 'cropped_chandak_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Microsoft model on Chandak-110 data',
        'reads_path': 'cropped_chandak_110/reads.txt',
        'gt_path': 'cropped_chandak_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Pretrained 110nt on Chandak-110 data',
        'reads_path': 'cropped_chandak_110/reads.txt',
        'gt_path': 'cropped_chandak_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on Chandak-110 data',
        'reads_path': 'cropped_chandak_110/reads.txt',
        'gt_path': 'cropped_chandak_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # Chandak-60 data (cropped to 60nt)
    # =============================================================
    {
        'name': 'Chandak model on Chandak-60 data',
        'reads_path': 'cropped_chandak_60/reads.txt',
        'gt_path': 'cropped_chandak_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Noisy DNA model on Chandak-60 data',
        'reads_path': 'cropped_chandak_60/reads.txt',
        'gt_path': 'cropped_chandak_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 60nt on Chandak-60 data',
        'reads_path': 'cropped_chandak_60/reads.txt',
        'gt_path': 'cropped_chandak_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on Chandak-60 data',
        'reads_path': 'cropped_chandak_60/reads.txt',
        'gt_path': 'cropped_chandak_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # Microsoft data (original 110nt)
    # =============================================================
    {
        'name': 'Microsoft model on Microsoft data',
        'reads_path': '../data/microsoft_data/data_microsoft/reads.txt',
        'gt_path': '../data/microsoft_data/data_microsoft/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Chandak model on Microsoft data',
        'reads_path': '../data/microsoft_data/data_microsoft/reads.txt',
        'gt_path': '../data/microsoft_data/data_microsoft/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Noisy DNA model on Microsoft data',
        'reads_path': '../data/microsoft_data/data_microsoft/reads.txt',
        'gt_path': '../data/microsoft_data/data_microsoft/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 110nt on Microsoft data',
        'reads_path': '../data/microsoft_data/data_microsoft/reads.txt',
        'gt_path': '../data/microsoft_data/data_microsoft/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Pretrained 60nt on Microsoft data',
        'reads_path': '../data/microsoft_data/data_microsoft/reads.txt',
        'gt_path': '../data/microsoft_data/data_microsoft/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on Microsoft data',
        'reads_path': '../data/microsoft_data/data_microsoft/reads.txt',
        'gt_path': '../data/microsoft_data/data_microsoft/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # Microsoft-60 data (cropped to 60nt)
    # =============================================================
    {
        'name': 'Noisy DNA model on Microsoft-60 data',
        'reads_path': 'cropped_microsoft_60/reads.txt',
        'gt_path': 'cropped_microsoft_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 60nt on Microsoft-60 data',
        'reads_path': 'cropped_microsoft_60/reads.txt',
        'gt_path': 'cropped_microsoft_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on Microsoft-60 data',
        'reads_path': 'cropped_microsoft_60/reads.txt',
        'gt_path': 'cropped_microsoft_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # Noisy DNA data (original 60nt)
    # =============================================================
    {
        'name': 'Noisy DNA model on Noisy DNA data',
        'reads_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/reads.txt',
        'gt_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Microsoft model on Noisy DNA data',
        'reads_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/reads.txt',
        'gt_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Chandak model on Noisy DNA data',
        'reads_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/reads.txt',
        'gt_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Pretrained 110nt on Noisy DNA data',
        'reads_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/reads.txt',
        'gt_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Pretrained 60nt on Noisy DNA data',
        'reads_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/reads.txt',
        'gt_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on Noisy DNA data',
        'reads_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/reads.txt',
        'gt_path': '../data/noisy_dna/noisy_dna_data_storage/data/splits/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },

    # =============================================================
    # DNAformer FC1 data (original 128nt)
    # =============================================================
    {
        'name': 'Microsoft model on DNAformer FC1 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Chandak model on DNAformer FC1 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Noisy DNA model on DNAformer FC1 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 110nt on DNAformer FC1 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer FC1 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    {
        'name': 'Pretrained 60nt on DNAformer FC1 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeFirstFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer FC1-110 data (cropped to 110nt)
    # =============================================================
    {
        'name': 'Microsoft model on DNAformer FC1-110 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Pretrained 110nt on DNAformer FC1-110 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer FC1-110 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer FC1-60 data (cropped to 60nt)
    # =============================================================
    {
        'name': 'Noisy DNA model on DNAformer FC1-60 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 60nt on DNAformer FC1-60 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer FC1-60 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeFirstFlowcell_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer FC2 data (original 128nt)
    # =============================================================
    {
        'name': 'Microsoft model on DNAformer FC2 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Chandak model on DNAformer FC2 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Noisy DNA model on DNAformer FC2 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 110nt on DNAformer FC2 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer FC2 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    {
        'name': 'Pretrained 60nt on DNAformer FC2 data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedNanoporeSecondFlowcell/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer FC2-110 data (cropped to 110nt)
    # =============================================================
    {
        'name': 'Microsoft model on DNAformer FC2-110 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Pretrained 110nt on DNAformer FC2-110 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer FC2-110 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer FC2-60 data (cropped to 60nt)
    # =============================================================
    {
        'name': 'Noisy DNA model on DNAformer FC2-60 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 60nt on DNAformer FC2-60 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer FC2-60 data',
        'reads_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedNanoporeSecondFlowcell_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer Pilot data (original 128nt)
    # =============================================================
    {
        'name': 'Microsoft model on DNAformer Pilot data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Chandak model on DNAformer Pilot data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 117,
        'model_variant': 'chandak',
    },
    {
        'name': 'Noisy DNA model on DNAformer Pilot data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 110nt on DNAformer Pilot data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer Pilot data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    {
        'name': 'Pretrained 60nt on DNAformer Pilot data',
        'reads_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/reads.txt',
        'gt_path': '../data/dnaformer/without_index_data/BinnedPilotNanopore/ground_truth.txt',
        'seq_length': 128,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer Pilot-110 data (cropped to 110nt)
    # =============================================================
    {
        'name': 'Microsoft model on DNAformer Pilot-110 data',
        'reads_path': 'cropped_dnaformer_BinnedPilotNanopore_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedPilotNanopore_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'microsoft',
    },
    {
        'name': 'Pretrained 110nt on DNAformer Pilot-110 data',
        'reads_path': 'cropped_dnaformer_BinnedPilotNanopore_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedPilotNanopore_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 110,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer Pilot-110 data',
        'reads_path': 'cropped_dnaformer_BinnedPilotNanopore_110/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedPilotNanopore_110/ground_truth.txt',
        'seq_length': 110,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
    # =============================================================
    # DNAformer Pilot-60 data (cropped to 60nt)
    # =============================================================
    {
        'name': 'Noisy DNA model on DNAformer Pilot-60 data',
        'reads_path': 'cropped_dnaformer_BinnedPilotNanopore_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedPilotNanopore_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'noisy_dna',
    },
    {
        'name': 'Pretrained 60nt on DNAformer Pilot-60 data',
        'reads_path': 'cropped_dnaformer_BinnedPilotNanopore_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedPilotNanopore_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 60,
        'model_variant': 'pretrained',
    },
    {
        'name': 'Variable length on DNAformer Pilot-60 data',
        'reads_path': 'cropped_dnaformer_BinnedPilotNanopore_60/reads.txt',
        'gt_path': 'cropped_dnaformer_BinnedPilotNanopore_60/ground_truth.txt',
        'seq_length': 60,
        'model_seq_length': 'var_50_120',
        'model_variant': 'pretrained',
    },
]

# Verify all data files exist
for exp in cross_experiments:
    for key in ['reads_path', 'gt_path']:
        if not os.path.exists(exp[key]):
            raise FileNotFoundError(f"{exp[key]} not found for experiment '{exp['name']}'")

print(f"Total experiments: {len(cross_experiments)}")

# Group and print by dataset
current_group = None
for exp in cross_experiments:
    # Detect group from reads_path
    rp = exp['reads_path']
    if 'chandak_110' in rp:
        group = 'Chandak-110 (cropped to 110nt)'
    elif 'chandak_60' in rp:
        group = 'Chandak-60 (cropped to 60nt)'
    elif 'chandak' in rp:
        group = 'Chandak (original 117nt)'
    elif 'microsoft_60' in rp:
        group = 'Microsoft-60 (cropped to 60nt)'
    elif 'microsoft' in rp:
        group = 'Microsoft (original 110nt)'
    elif 'noisy_dna' in rp:
        group = 'Noisy DNA (original 60nt)'
    elif 'FirstFlowcell_110' in rp:
        group = 'DNAformer FC1-110 (cropped to 110nt)'
    elif 'FirstFlowcell_60' in rp:
        group = 'DNAformer FC1-60 (cropped to 60nt)'
    elif 'FirstFlowcell' in rp:
        group = 'DNAformer FC1 (original 128nt)'
    elif 'SecondFlowcell_110' in rp:
        group = 'DNAformer FC2-110 (cropped to 110nt)'
    elif 'SecondFlowcell_60' in rp:
        group = 'DNAformer FC2-60 (cropped to 60nt)'
    elif 'SecondFlowcell' in rp:
        group = 'DNAformer FC2 (original 128nt)'
    elif 'PilotNanopore_110' in rp:
        group = 'DNAformer Pilot-110 (cropped to 110nt)'
    elif 'PilotNanopore_60' in rp:
        group = 'DNAformer Pilot-60 (cropped to 60nt)'
    elif 'PilotNanopore' in rp:
        group = 'DNAformer Pilot (original 128nt)'
    else:
        group = 'Other'
    if group != current_group:
        current_group = group
        print(f"\n  {group}:")
    print(f"    - {exp['name']}")
print("\nAll data files found.")

Total experiments: 65

  Chandak (original 117nt):
    - Chandak model on Chandak data
    - Microsoft model on Chandak data
    - Noisy DNA model on Chandak data
    - Pretrained 110nt on Chandak data
    - Pretrained 60nt on Chandak data
    - Variable length on Chandak data

  Chandak-110 (cropped to 110nt):
    - Chandak model on Chandak-110 data
    - Microsoft model on Chandak-110 data
    - Pretrained 110nt on Chandak-110 data
    - Variable length on Chandak-110 data

  Chandak-60 (cropped to 60nt):
    - Chandak model on Chandak-60 data
    - Noisy DNA model on Chandak-60 data
    - Pretrained 60nt on Chandak-60 data
    - Variable length on Chandak-60 data

  Microsoft (original 110nt):
    - Microsoft model on Microsoft data
    - Chandak model on Microsoft data
    - Noisy DNA model on Microsoft data
    - Pretrained 110nt on Microsoft data
    - Pretrained 60nt on Microsoft data
    - Variable length on Microsoft data

  Microsoft-60 (cropped to 60nt):
    - Noisy DNA mode

## 4. Run Inference

In [7]:
# Load cached results (skip already-completed experiments)
import pickle
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)
cross_pkl = results_dir / 'cross_results.pkl'
if cross_pkl.exists():
    with open(cross_pkl, 'rb') as f:
        cross_results = pickle.load(f)
    print(f"Loaded {len(cross_results)} cached experiments from {cross_pkl}")
else:
    cross_results = {}

# Maximum number of examples to inference on per experiment (set None for no limit)
MAX_EXAMPLES = 15000

stoi, itos = load_vocabulary()
decode = lambda t: ''.join(itos[i] for i in t)
encode_str = lambda s: [stoi.get(ch, stoi.get('<unk>', 0)) for ch in s]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ctx = torch.amp.autocast('cuda', dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

for exp in cross_experiments:
    print(f"\n{'='*60}")
    print(f"Experiment: {exp['name']}")
    print(f"{'='*60}")

    if exp['name'] in cross_results:
        print(f"  CACHED ({len(cross_results[exp['name']])} examples) — skipping")
        continue

    # --- Load data ---
    with open(exp['reads_path'], 'r') as f:
        content = f.read()
    clusters = content.split('===============================')
    clusters = [c.strip().split('\n') for c in clusters if c.strip()]
    valid_clusters = [c for c in clusters if 2 <= len(c) <= 10]

    with open(exp['gt_path'], 'r') as f:
        all_gts = [line.strip() for line in f if line.strip()]
    valid_indices = [i for i, c in enumerate(clusters) if 2 <= len(c) <= 10]
    gts = [all_gts[i] for i in valid_indices]

    exp_inputs = []
    for cluster, gt in zip(valid_clusters, gts):
        reads_str = '|'.join(cluster)
        exp_inputs.append(f"{reads_str}:{gt}")

    # --- Load model (use model_seq_length to look up the correct model) ---
    exp_model_info = get_model_info(exp['model_seq_length'], exp['model_variant'])
    print(f"  Model: {exp_model_info['model_name']} ({exp_model_info['description']})")

    # Skip examples that exceed the model's context window
    block_size = exp_model_info['block_size']
    filtered_inputs = []
    filtered_gts = []
    for inp, gt in zip(exp_inputs, gts):
        input_len = len(inp.split(':')[0]) + 1 + exp['seq_length']
        if input_len > block_size:
            continue
        filtered_inputs.append(inp)
        filtered_gts.append(gt)
    if len(filtered_inputs) < len(exp_inputs):
        print(f"  Skipped {len(exp_inputs) - len(filtered_inputs)}/{len(exp_inputs)} examples exceeding block_size={block_size}")
    exp_inputs = filtered_inputs
    gts = filtered_gts

    # Cap the number of examples for inference
    if MAX_EXAMPLES is not None and len(exp_inputs) > MAX_EXAMPLES:
        print(f"  Capping from {len(exp_inputs)} to {MAX_EXAMPLES} examples")
        exp_inputs = exp_inputs[:MAX_EXAMPLES]
        gts = gts[:MAX_EXAMPLES]

    print(f"  Loaded {len(exp_inputs)} test examples")

    if len(exp_inputs) == 0:
        print("  No valid examples — skipping")
        continue

    model_path = hf_hub_download(
        repo_id=exp_model_info['repo_id'],
        filename=exp_model_info['model_name'],
        cache_dir='./models'
    )

    ckpt = torch.load(model_path, map_location='cpu')
    config_args = {k: v for k, v in ckpt['model_args'].items() if k != 'model_type'}
    state_dict = ckpt['model']
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key.replace('_orig_mod.', '') if key.startswith('_orig_mod.') else key
        new_state_dict[new_key] = value

    exp_model = GPT(GPTConfig(**config_args))
    exp_model.load_state_dict(new_state_dict, strict=True)
    exp_model = exp_model.half().to(device).eval()

    # --- Run inference ---
    inf_params = {
        'model': exp_model,
        'ctx': ctx,
        'device': device,
        'stoi': stoi,
        'itos': itos,
        'encode': encode_str,
        'decode': decode,
        'temperature': 1.0,
        'greedy': True,
        'ground_truth_length': exp['seq_length'],
        'block_size': exp_model_info['block_size'],
        'target_type': 'CPRED',
        'constrained_generation': True
    }

    # Sort by length for efficient batching
    examples_with_len = []
    for idx, (inp, gt) in enumerate(zip(exp_inputs, gts)):
        examples_with_len.append((idx, inp, gt, len(inp.split(':')[0])))
    examples_with_len.sort(key=lambda x: x[3])

    batch_size = 200
    sorted_results = []
    with torch.inference_mode(), ctx:
        for start_idx in tqdm(range(0, len(examples_with_len), batch_size), desc=f"  Inference"):
            batch = examples_with_len[start_idx:start_idx + batch_size]
            batch_inputs = [ex[1] for ex in batch]
            batch_gts = [ex[2] for ex in batch]
            alignment_sizes = [len(inp.split(':')[0].split('|')) for inp in batch_inputs]

            out = GPT_Inference(inf_params).inference(batch_inputs, alignment_size=alignment_sizes)
            predictions = out['candidate_sequences']

            for orig_idx, inp, gt, pred, cs in zip(
                [ex[0] for ex in batch], batch_inputs, batch_gts, predictions, alignment_sizes
            ):
                pred_filtered = filter_string(pred)[:len(gt)]
                sorted_results.append({
                    'original_index': orig_idx,
                    'cluster_size': cs,
                    'ground_truth': gt,
                    'prediction': pred_filtered,
                    'hamming': hamming_distance_postprocessed(gt, pred_filtered),
                    'levenshtein': levenshtein_distance(gt, pred_filtered) / len(gt),
                })

    # Restore original order
    sorted_results.sort(key=lambda x: x['original_index'])
    exp_results = [{k: v for k, v in r.items() if k != 'original_index'} for r in sorted_results]
    cross_results[exp['name']] = exp_results

    # Print summary
    lev = [r['levenshtein'] for r in exp_results]
    fail = sum(l > 0 for l in lev)
    print(f"  Overall Mean Levenshtein: {np.mean(lev):.4f} +/- {np.std(lev):.4f}")
    print(f"  Overall Failure Rate: {fail}/{len(lev)} ({100*fail/len(lev):.1f}%)")
    print(f"  {'Size':<6} {'Count':<8} {'Mean Lev':<12} {'Failure %':<10}")
    by_cs = defaultdict(list)
    for r in exp_results:
        by_cs[r['cluster_size']].append(r)
    for size in sorted(by_cs.keys()):
        rs = by_cs[size]
        l = [r['levenshtein'] for r in rs]
        fail_pct = 100 * sum(v > 0 for v in l) / len(l)
        print(f"  {size:<6} {len(rs):<8} {np.mean(l):<12.4f} {fail_pct:<10.1f}%")

    # Save after each experiment (incremental caching)
    with open(cross_pkl, 'wb') as f:
        pickle.dump(cross_results, f)

    # Free GPU memory
    del exp_model, ckpt, state_dict, new_state_dict
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("All cross-dataset experiments complete!")
print(f"{'='*60}")



Experiment: Chandak model on Chandak data
  Model: finetuned_chandak_len117.pt (Fine-tuned on Chandak dataset (117nt))
  Loaded 8481 test examples
FlashAttention available (will be used during training and inference with KV cache)


  Inference: 100%|██████████| 43/43 [02:15<00:00,  3.15s/it]


  Overall Mean Levenshtein: 0.0687 +/- 0.1176
  Overall Failure Rate: 5601/8481 (66.0%)
  Size   Count    Mean Lev     Failure % 
  2      1273     0.1608       99.4      %
  3      1054     0.1231       94.2      %
  4      1029     0.0717       81.9      %
  5      932      0.0547       70.1      %
  6      868      0.0410       59.6      %
  7      865      0.0338       49.2      %
  8      836      0.0293       43.3      %
  9      803      0.0216       34.4      %
  10     821      0.0201       32.4      %

Experiment: Microsoft model on Chandak data
  Model: finetuned_microsoft_dna_len110.pt (Fine-tuned on Microsoft DNA dataset (110nt))
  Loaded 8481 test examples


  Inference: 100%|██████████| 43/43 [02:33<00:00,  3.56s/it]


  Overall Mean Levenshtein: 0.1365 +/- 0.1157
  Overall Failure Rate: 8478/8481 (100.0%)
  Size   Count    Mean Lev     Failure % 
  2      1273     0.2115       100.0     %
  3      1054     0.1672       100.0     %
  4      1029     0.1296       100.0     %
  5      932      0.1177       100.0     %
  6      868      0.1044       100.0     %
  7      865      0.0957       99.9      %
  8      836      0.0933       99.9      %
  9      803      0.0761       99.9      %
  10     821      0.1912       100.0     %

Experiment: Noisy DNA model on Chandak data
  Model: finetuned_noisy_dna_len60.pt (Fine-tuned on Noisy DNA dataset (60nt))
  Skipped 4184/8481 examples exceeding block_size=800
  Loaded 4297 test examples


...

  Overall Mean Levenshtein: 0.0157 +/- 0.0285
  Overall Failure Rate: 5899/15000 (39.3%)
  Size   Count    Mean Lev     Failure % 
  2      1745     0.0503       76.7      %
  3      1659     0.0242       57.6      %
  4      1688     0.0167       46.6      %
  5      1675     0.0121       37.7      %
  6      1620     0.0097       32.9      %
  7      1647     0.0081       29.6      %
  8      1608     0.0068       25.1      %
  9      1707     0.0062       23.8      %
  10     1651     0.0054       21.6      %

Experiment: Pretrained 60nt on DNAformer Pilot-60 data
  Model: model_seq_len_60.pt (Pretrained on synthetic IDS data (60nt))
  Capping from 126377 to 15000 examples
  Loaded 15000 test examples


  Inference: 100%|██████████| 75/75 [01:21<00:00,  1.09s/it]


  Overall Mean Levenshtein: 0.0048 +/- 0.0173
  Overall Failure Rate: 1772/15000 (11.8%)
  Size   Count    Mean Lev     Failure % 
  2      1745     0.0293       59.1      %
  3      1659     0.0066       20.2      %
  4      1688     0.0024       9.0       %
  5      1675     0.0012       4.9       %
  6      1620     0.0009       4.1       %
  7      1647     0.0005       2.5       %
  8      1608     0.0005       2.1       %
  9      1707     0.0002       1.1       %
  10     1651     0.0001       0.6       %

Experiment: Variable length on DNAformer Pilot-60 data
  Model: model_var_len_50_120.pt (Pretrained on variable length sequences (50-120nt). Best for fine-tuning on custom real-world data.)
  Capping from 126377 to 15000 examples
  Loaded 15000 test examples


  Inference: 100%|██████████| 75/75 [01:21<00:00,  1.09s/it]


  Overall Mean Levenshtein: 0.0104 +/- 0.0298
  Overall Failure Rate: 2660/15000 (17.7%)
  Size   Count    Mean Lev     Failure % 
  2      1745     0.0601       77.5      %
  3      1659     0.0130       29.5      %
  4      1688     0.0059       15.6      %
  5      1675     0.0043       11.8      %
  6      1620     0.0026       7.6       %
  7      1647     0.0018       5.0       %
  8      1608     0.0014       4.4       %
  9      1707     0.0010       3.0       %
  10     1651     0.0007       1.9       %

All cross-dataset experiments complete!


## 5. Results Comparison

In [8]:
# Compare cross-dataset results side by side
print("CROSS-DATASET COMPARISON")
print("="*90)
print(f"{'Experiment':<40} {'Mean Lev':<12} {'Failure %':<12} {'N':<8}")
print("-"*90)

for name, results in cross_results.items():
    lev = [r['levenshtein'] for r in results]
    fail_pct = 100 * sum(l > 0 for l in lev) / len(lev)
    print(f"{name:<40} {np.mean(lev):<12.4f} {fail_pct:<12.1f} {len(results):<8}")

print("="*90)

# Performance by cluster size for each experiment
for name, results in cross_results.items():
    print(f"\n{name}")
    print("-"*60)
    print(f"  {'Size':<6} {'Count':<8} {'Mean Lev':<12} {'Failure %':<10}")

    by_cs = defaultdict(list)
    for r in results:
        by_cs[r['cluster_size']].append(r)

    for size in sorted(by_cs.keys()):
        rs = by_cs[size]
        l = [r['levenshtein'] for r in rs]
        fail_pct = 100 * sum(v > 0 for v in l) / len(l)
        print(f"  {size:<6} {len(rs):<8} {np.mean(l):<12.4f} {fail_pct:<10.1f}%")

CROSS-DATASET COMPARISON
Experiment                               Mean Lev     Failure %    N       
------------------------------------------------------------------------------------------
Chandak model on Chandak data            0.0687       66.0         8481    
Microsoft model on Chandak data          0.1365       100.0        8481    
Noisy DNA model on Chandak data          0.3391       100.0        4297    
Pretrained 110nt on Chandak data         0.1503       100.0        8481    
Pretrained 60nt on Chandak data          0.3150       100.0        4297    
Variable length on Chandak data          0.0912       90.6         8481    
Chandak model on Chandak-110 data        0.0735       68.0         8481    
Microsoft model on Chandak-110 data      0.0788       83.9         8481    
Pretrained 110nt on Chandak-110 data     0.0755       82.7         8481    
Variable length on Chandak-110 data      0.0931       89.5         8481    
Chandak model on Chandak-60 data         0.2912 

## 6. Plots

In [9]:
# Plot 1: Cross-dataset overview. All models on each original dataset.
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)

dataset_groups = {
    'Chandak (117nt)': {name: res for name, res in cross_results.items()
                        if 'Chandak data' in name and 'Chandak-110' not in name and 'Chandak-60' not in name},
    'Microsoft (110nt)': {name: res for name, res in cross_results.items()
                          if 'Microsoft data' in name and 'Microsoft-60' not in name},
    'Noisy DNA (60nt)': {name: res for name, res in cross_results.items()
                         if 'Noisy DNA data' in name},
    'DNAformer FC1 (128nt)': {name: res for name, res in cross_results.items()
                              if 'DNAformer FC1 data' in name and 'FC1-110' not in name},
    'DNAformer FC2 (128nt)': {name: res for name, res in cross_results.items()
                              if 'DNAformer FC2 data' in name and 'FC2-110' not in name},
    'DNAformer Pilot (128nt)': {name: res for name, res in cross_results.items()
                                if 'DNAformer Pilot data' in name and 'Pilot-110' not in name},
}

# Distinct color + marker + linestyle for each model
model_style = {
    'Microsoft model':   {'color': '#1f77b4', 'marker': 's', 'ls': '-',  'lw': 2.0},
    'Chandak model':     {'color': '#ff7f0e', 'marker': '^', 'ls': '-',  'lw': 2.0},
    'Noisy DNA model':   {'color': '#2ca02c', 'marker': 'D', 'ls': '-',  'lw': 2.0},
    'Pretrained 110nt':  {'color': '#d62728', 'marker': 'v', 'ls': '--', 'lw': 1.5},
    'Pretrained 60nt':   {'color': '#9467bd', 'marker': 'x', 'ls': '--', 'lw': 1.5},
    'Variable length':   {'color': '#8c564b', 'marker': 'P', 'ls': ':',  'lw': 1.5},
}

def get_style(name):
    label = name.split(' on ')[0]
    for key, style in model_style.items():
        if label.startswith(key):
            return label, style
    return label, {'color': 'gray', 'marker': 'o', 'ls': '-', 'lw': 1.0}

fig, axes = plt.subplots(2, 6, figsize=(36, 9))

for col, (title, exps) in enumerate(dataset_groups.items()):
    for name, results in sorted(exps.items()):
        by_cs = defaultdict(list)
        for r in results:
            by_cs[r['cluster_size']].append(r)
        sizes = sorted(by_cs.keys())
        mean_lev = [np.mean([r['levenshtein'] for r in by_cs[s]]) for s in sizes]
        fail_rate = [100 * sum(r['levenshtein'] > 0 for r in by_cs[s]) / len(by_cs[s]) for s in sizes]

        label, style = get_style(name)
        for row, yvals in [(0, mean_lev), (1, fail_rate)]:
            axes[row, col].plot(sizes, yvals,
                                color=style['color'], marker=style['marker'],
                                linestyle=style['ls'], linewidth=style['lw'],
                                markersize=6, label=label)

    for row in range(2):
        axes[row, col].set_xlabel('Cluster Size')
        axes[row, col].legend(fontsize=8, loc='best')
        axes[row, col].grid(True, alpha=0.3)
    axes[0, col].set_yscale('log')
    axes[0, col].set_ylabel('Mean Levenshtein Distance')
    axes[0, col].set_title(f'{title}')
    axes[1, col].set_ylabel('Failure Rate (%)')
    axes[1, col].set_title(f'{title}')

plt.suptitle('Cross-Dataset Transfer: All Models on Original Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(results_dir / 'cross_dataset_overview.png', bbox_inches='tight', dpi=150)
plt.savefig(results_dir / 'cross_dataset_overview.pdf', bbox_inches='tight', dpi=300)
print(f'Saved to results/cross_dataset_overview.pdf')
plt.show()


Saved to results/cross_dataset_overview.pdf


In [10]:
# Plot 2: Length-controlled comparison. Only show models that exist in both original and cropped.
fig, axes = plt.subplots(2, 9, figsize=(54, 9))

comparisons = [
    {
        'title': 'Chandak: 117nt vs 110nt',
        'original': {n: r for n, r in cross_results.items() if 'Chandak data' in n and 'Chandak-110' not in n and 'Chandak-60' not in n},
        'cropped': {n: r for n, r in cross_results.items() if 'Chandak-110' in n},
        'crop_label': '110nt',
    },
    {
        'title': 'Chandak: 117nt vs 60nt',
        'original': {n: r for n, r in cross_results.items() if 'Chandak data' in n and 'Chandak-110' not in n and 'Chandak-60' not in n},
        'cropped': {n: r for n, r in cross_results.items() if 'Chandak-60' in n},
        'crop_label': '60nt',
    },
    {
        'title': 'Microsoft: 110nt vs 60nt',
        'original': {n: r for n, r in cross_results.items() if 'Microsoft data' in n and 'Microsoft-60' not in n},
        'cropped': {n: r for n, r in cross_results.items() if 'Microsoft-60' in n},
        'crop_label': '60nt',
    },
    {
        'title': 'DNAformer FC1: 128nt vs 110nt',
        'original': {n: r for n, r in cross_results.items() if 'DNAformer FC1 data' in n and 'FC1-110' not in n},
        'cropped': {n: r for n, r in cross_results.items() if 'DNAformer FC1-110' in n},
        'crop_label': '110nt',
    },
    {
        'title': 'DNAformer FC2: 128nt vs 110nt',
        'original': {n: r for n, r in cross_results.items() if 'DNAformer FC2 data' in n and 'FC2-110' not in n},
        'cropped': {n: r for n, r in cross_results.items() if 'DNAformer FC2-110' in n},
        'crop_label': '110nt',
    },
    {
        'title': 'DNAformer Pilot: 128nt vs 110nt',
        'original': {n: r for n, r in cross_results.items() if 'DNAformer Pilot data' in n and 'Pilot-110' not in n},
        'cropped': {n: r for n, r in cross_results.items() if 'DNAformer Pilot-110' in n},
        'crop_label': '110nt',
    },
]

for col, comp in enumerate(comparisons):
    orig_models = {n.split(' on ')[0]: (n, r) for n, r in comp['original'].items()}
    crop_models = {n.split(' on ')[0]: (n, r) for n, r in comp['cropped'].items()}

    # Only show models that exist in the cropped group
    models_to_show = sorted(crop_models.keys())

    for i, model in enumerate(models_to_show):
        style = model_style.get(model, {'color': 'gray', 'marker': 'o', 'ls': '-', 'lw': 1.5})
        c = style['color']
        mk = style['marker']

        # Original (dashed, faded)
        if model in orig_models:
            name, results = orig_models[model]
            by_cs = defaultdict(list)
            for r in results:
                by_cs[r['cluster_size']].append(r)
            sizes = sorted(by_cs.keys())
            mean_lev = [np.mean([r['levenshtein'] for r in by_cs[s]]) for s in sizes]
            fail_rate = [100 * sum(r['levenshtein'] > 0 for r in by_cs[s]) / len(by_cs[s]) for s in sizes]
            axes[0, col].plot(sizes, mean_lev, marker=mk, linestyle='--', color=c, linewidth=1.5, markersize=4, alpha=0.4, label=f'{model} (orig)')
            axes[1, col].plot(sizes, fail_rate, marker=mk, linestyle='--', color=c, linewidth=1.5, markersize=4, alpha=0.4, label=f'{model} (orig)')

        # Cropped (solid)
        if model in crop_models:
            name, results = crop_models[model]
            by_cs = defaultdict(list)
            for r in results:
                by_cs[r['cluster_size']].append(r)
            sizes = sorted(by_cs.keys())
            mean_lev = [np.mean([r['levenshtein'] for r in by_cs[s]]) for s in sizes]
            fail_rate = [100 * sum(r['levenshtein'] > 0 for r in by_cs[s]) / len(by_cs[s]) for s in sizes]
            axes[0, col].plot(sizes, mean_lev, marker=mk, linestyle='-', color=c, linewidth=2, markersize=6, label=f'{model} ({comp["crop_label"]})')
            axes[1, col].plot(sizes, fail_rate, marker=mk, linestyle='-', color=c, linewidth=2, markersize=6, label=f'{model} ({comp["crop_label"]})')

    axes[0, col].set_yscale('log')
    axes[0, col].set_xlabel('Cluster Size')
    axes[0, col].set_ylabel('Mean Levenshtein Distance')
    axes[0, col].set_title(comp['title'])
    axes[0, col].legend(fontsize=7)
    axes[0, col].grid(True, alpha=0.3)
    axes[1, col].set_xlabel('Cluster Size')
    axes[1, col].set_ylabel('Failure Rate (%)')
    axes[1, col].set_title(comp['title'])
    axes[1, col].legend(fontsize=7)
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Length-Controlled: Original vs Cropped (dashed=original, solid=cropped)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(results_dir / 'length_controlled_comparison.png', bbox_inches='tight', dpi=150)
plt.savefig(results_dir / 'length_controlled_comparison.pdf', bbox_inches='tight', dpi=300)
print(f'Saved to results/length_controlled_comparison.pdf')
plt.show()


Saved to results/length_controlled_comparison.pdf


In [11]:
# Plot 3: Summary bar chart. Overall mean Levenshtein per experiment, grouped by dataset.
fig, axes = plt.subplots(1, 9, figsize=(54, 6))

dataset_order = [
    ('Chandak', ['Chandak data', 'Chandak-110', 'Chandak-60']),
    ('Microsoft', ['Microsoft data', 'Microsoft-60']),
    ('Noisy DNA', ['Noisy DNA data']),
    ('DNAformer FC1', ['DNAformer FC1 data', 'DNAformer FC1-110', 'DNAformer FC1-60']),
    ('DNAformer FC2', ['DNAformer FC2 data', 'DNAformer FC2-110', 'DNAformer FC2-60']),
    ('DNAformer Pilot', ['DNAformer Pilot data', 'DNAformer Pilot-110', 'DNAformer Pilot-60']),
]

for ax_idx, (ds_name, suffixes) in enumerate(dataset_order):
    labels = []
    values = []
    bar_colors = []
    
    for suffix in suffixes:
        matching = {n: r for n, r in cross_results.items() if suffix in n}
        for name in sorted(matching.keys()):
            results = matching[name]
            mean_lev = np.mean([r['levenshtein'] for r in results])
            labels.append(name.replace(' data', '').replace(' on ', '\n→ '))
            values.append(mean_lev)
            if 'Pretrained' in name or 'Variable' in name:
                bar_colors.append('tab:gray')
            elif ds_name.lower() in name.lower().split(' on ')[0].lower().replace('microsoft','microsoft').replace('chandak','chandak'):
                bar_colors.append('tab:blue')  # in-domain
            else:
                bar_colors.append('tab:orange')  # cross-domain
    
    y_pos = np.arange(len(labels))
    ax = axes[ax_idx]
    bars = ax.barh(y_pos, values, color=['tab:blue' if v == min(values) else 'tab:orange' for v in values], alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xscale('log')
    ax.set_xlabel('Mean Levenshtein Distance')
    ax.set_title(f'{ds_name} Datasets')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()

plt.suptitle('Overall Performance Summary (lower = better)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(results_dir / 'summary_bar_chart.png', bbox_inches='tight', dpi=150)
plt.savefig(results_dir / 'summary_bar_chart.pdf', bbox_inches='tight', dpi=300)
print(f'Saved to results/summary_bar_chart.png')
plt.show()


Saved to results/summary_bar_chart.png
